# Exploratory Data Analysis

This notebook presents an exploratory data analysis (EDA) of the MVTec Anomaly Detection [(MVTec AD)](https://www.mvtec.com/research-teaching/datasets/mvtec-ad) dataset. MVTec AD is a benchmark dataset for unsupervised anomaly detection in industrial inspection and contains over 5,000 images across 15 object categories. Each category contains defect-free training images and a test set consisting of both normal and anomalous samples.

The purpose of this analysis is to:

1. Identify the object and texture categories present within the dataset.
2. Examine the distribution of normal and anomalous samples.
3. Investigate the defect types available within each category.
4. Analyse image characteristics such as resolution, brightness, sharpness, and contrast.
5. Identify potential challenges and limitations that may influence anomaly detection performance.
6. Inform preprocessing and model design decisions for the subsequent stages of the project.

In [ ]:
import os 

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

import cv2

from pathlib import Path

ROOT = Path.cwd().parents[1]

IMAGE_DIR = ROOT / "images/EDA"

In [ ]:
os.makedirs(IMAGE_DIR, exist_ok=True)

df = pd.read_csv("../../data/metadata.csv")

## 1. Identify the object and texture categories present within the dataset.

In [ ]:
categories = sorted(df["category"].unique())

print(len(categories))
categories

## 2. Examine the distribution of normal and anomalous samples.

This section examines how samples are distributed across the dataset. Understanding the balance between normal and anomalous images helps inform about any class imbalances may influence anomaly detection performance and evaluation metrics.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

defect_counts = (df["label"].map({0: "Normal", 1: "Defective"}).value_counts())

defect_counts.plot(
    kind="pie",
    autopct="%1.1f%%",
    ax=axes[0],
    colors=["#66b3ff", "#ff9999"]
)

df["split"].value_counts().plot(
    kind="pie",
    autopct="%1.1f%%",
    ax=axes[1]
)

axes[0].set_title("Normal vs Defective")

axes[1].set_title("Train vs Test")

plt.tight_layout()
plt.savefig(f'{IMAGE_DIR}/class_distribution_pie.svg', bbox_inches="tight", pad_inches=0.1)
plt.show()

### Category Distribution

This investigates how images are distributed across the 15 MVTec categories. Identifying category-level imbalances helps determine whether certain categories may be overrepresented during evaluation.

In [ ]:
df["category"].value_counts().plot(kind="pie", autopct="%1.1f%%")
plt.title("Category Distribution")


plt.savefig(f'{IMAGE_DIR}/category_distribution_pie.svg', bbox_inches="tight", pad_inches=0.1)
plt.show()

The dataset is reasonably balanced across categories, although some categories contain more images than others. No category dominates the dataset to a degree that would significantly bias overall evaluation.

#

This figure shows the proportion of normal and defective samples within each category. As MVTec is designed for anomaly detection, categories may contain different ratios of normal and anomalous images depending on the number of defect types available.

In [ ]:
ax = (df.groupby("category")["label"].value_counts(normalize=True).unstack().plot(kind="bar", stacked=True))

ax.legend(title="Label", labels=["Normal", "Defective"], bbox_to_anchor=(1, 1.3), loc="upper right")

plt.title("Defect Distribution by Category")

plt.xlabel("Category")

plt.ylabel("Proportion")
plt.yticks(np.arange(0, 1.1, 0.1))

plt.savefig(f'{IMAGE_DIR}/defect_distribution_category_chart.svg', bbox_inches="tight", pad_inches=0.1)
plt.show()

The proportion of defective samples varies between categories. Categories containing a larger number of defect types generally show a higher proportion of anomalous test samples.

## 3. Investigate the defect types available within each category.

Understanding the variety of defect types present within each category is for assessing dataset complexity. Categories containing a larger number of defect types may present a more challenging anomaly detection task.

The following visualisations show how defect types are distributed within each category's test set. This provides insight into the relative frequency of different anomaly types.

In [ ]:
###
# Something I can come back too if embeddings put scratches/contamination in the same cluster together across different 
# categories should probbaly make those defects share colour
###

fig, axes = plt.subplots(3, 5, figsize=(30, 18))

axes = axes.flatten()

fig.suptitle(
    "Distribution of Good vs Defective Images by Category (Test Split)",
    fontsize=16
)

for i, category in enumerate(sorted(df["category"].unique())):
    ax = axes[i]

    counts = (  # How man defects in each category
        df[
            (df["category"] == category)
            & (df["split"] == "test")
        ]["type"]
        .value_counts()
    )

    colors = list(plt.cm.tab20.colors[:len(counts)]) # Gets all colours from tab20 up to len(counts)

    for j, defect_type in enumerate(counts.index):
        if defect_type == "good":
            colors[j] = "#66b3ff"   # Ensures good is the same colour across all pies

    counts.plot(
        kind="pie",
        ax=ax,
        autopct="%1.1f%%",
        colors=colors
    )

    ax.set_title(category)

plt.tight_layout(rect=[0, 0, 1, 0.97]) # How much of the plot should be used to arrange subplots
plt.savefig(f'{IMAGE_DIR}/distibution_defects_category_pies.svg', bbox_inches="tight", pad_inches=0.1)
plt.show()

The number and distribution of defect types varies substantially between categories. Some categories contain only a small number of anomaly classes, while others contain a wider range of defect variations.

#

This analysis examines the number of unique defect types present within each category. A larger number of defect types may indicate increased anomaly diversity and greater detection complexity.

In [ ]:
cat_type = df.groupby("category")["type"].nunique()
max_types = cat_type.max()

df.groupby("category")["type"].nunique().sort_values().plot(kind="barh")

plt.title("Number of Unique Types of Defects per Category")

plt.ylabel("Category")

plt.xlabel("Number of Defects")
plt.xticks(range(0, max_types + 1))

plt.grid(axis='x', linestyle='--', alpha=0.7)

plt.savefig(f'{IMAGE_DIR}/defects_per_category_chart.svg', bbox_inches="tight", pad_inches=0.1)
plt.show()

The following figure provides representative examples of normal and defective samples for each category. Visual inspection helps identify differences in texture, appearance, and defect characteristics throughout the dataset.

In [ ]:
fig, axes = plt.subplots(
    len(categories),
    max_types + 1,
    figsize=(4 * max_types, 3 * len(categories))
)

for row, cat in enumerate(categories):
    # Randomally sampel one image of each type for the current category from the test split
    sample = (
        df[
            (df["category"] == cat)
            & (df["split"] == "test")
        ]
        .groupby("type", group_keys=False)
        .sample(n=1)
    )

    # Makes it so good is always in the first column
    sample = sample.sort_values("type", key=lambda s: s.map(lambda x: (x != "good", x)))

    axes[row, 0].text(
    0.5,
    0.5,
    cat.capitalize(),
    fontsize=16,
    fontweight="bold",
    ha="center",
    va="center"
    )

    axes[row, 0].axis("off")

    for col, (_, record) in enumerate(sample.iterrows(), start=1):

        ax = axes[row, col]

        img = plt.imread(ROOT / record["path"])

        ax.imshow(img)
        ax.set_title(record["type"])
        ax.axis("off")

    # hide unused columns
    for col in range(len(sample), max_types+1):
        axes[row, col].axis("off")

fig.suptitle(
    "Example MVTec Test Images by Category and Defect Type",
    fontsize=20,
    fontweight="bold"
)

plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.savefig(f'{IMAGE_DIR}/defect_examples.png', bbox_inches="tight", pad_inches=0.1)
plt.show()

## 4. Analyse image characteristics such as resolution, brightness, sharpness, and contrast.

This section investigates several image characteristics that may influence feature extraction and anomaly detection performance. Resolution, brightness, sharpness, and contrast are analysed to identify potential inconsistencies or preprocessing requirements.

Image-level statistics are computed for every image in the dataset. Brightness is measured as the mean pixel intensity, contrast as the standard deviation of pixel intensities, and sharpness using the variance of the Laplacian.

In [ ]:
stats = []

for row in df.itertuples():

    img = cv2.imread(str(ROOT / row.path), cv2.IMREAD_GRAYSCALE)

    h, w = img.shape

    stats.append({
        "width": w,
        "height": h,
        "brightness": img.mean(), #0-255
        "sharpness": cv2.Laplacian(img, cv2.CV_64F).var(), #2nd derivative of how much pixels change around eachother
        "contrast": img.std(),
        "category": row.category
    })

stats = pd.DataFrame(stats)

stats

In [ ]:
# Total statistics descriptions
print(stats[["width", "height"]].describe()) 
print("\n\nSize Value Counts:") 
print(stats[["width", "height"]].value_counts())

print("\n\nBrightness Description:") 
print(stats["brightness"].describe())

print("\n\nSharpness Description:")
print(stats["sharpness"].describe())

print("\n\nContrast Description:")
print(stats["contrast"].describe())

In [ ]:
# Find how image sizes are distributed per category
print(
    stats.groupby("category")
    .agg({
        "width": ["mean", "std"],
        "height": ["mean", "std"],
    })
)

In [ ]:
print(
    stats.groupby("category")
         .agg({
             "brightness": ["mean", "std", "min", "max"]
         })
)

print("\n\n",
    stats.groupby("category")
         .agg({
            "sharpness": ["mean", "std", "min", "max"]
         })
)

print("\n\n",
    stats.groupby("category")
         .agg({
            "contrast": ["mean", "std", "min", "max"]
         })
)     
            

### Observations

Several notable observations can be made from the image characteristic analysis:

- Image resolution is consistent within each category but varies between categories, ranging from 700×700 to 1024×1024 pixels.
- Brightness varies significantly between categories due to differences in object appearance and material properties, but remains highly consistent within individual categories.
- Sharpness and contrast exhibit greater variation between categories, reflecting differences in texture complexity and surface structure.
- Categories such as carpet, wood, and toothbrush contain substantially more texture detail than smoother categories such as capsule and screw.

These findings suggest that resizing and normalization will be necessary during preprocessing, while category-specific texture complexity may influence anomaly detection performance.